<a href="https://colab.research.google.com/github/bangaru01/C_programing/blob/main/TS_ring_pucker_CREST.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TS Ring-Pucker Conformer Search (CREST + Cremer–Pople)

Workflow:
1. Install `xtb` + `CREST`
2. Mount Google Drive (so you can read your 34 XYZ files and save results back)
3. Set which atoms define the ring and the forming bond
4. Run a **constrained** CREST conformer search (forming bond held fixed, everything else sampled freely)
5. Classify every conformer CREST finds (Chair / Boat / Twist-boat / Envelope / Half-chair) via formal Cremer–Pople analysis

**Important:** CREST here returns *constrained local minima*, not genuine transition states (no imaginary-frequency check).
Use these as starting geometries for real TS optimizations afterward — this step is for finding
which ring-pucker families are even worth optimizing.


## 1. Install xtb + CREST

In [13]:
!apt-get -qq update
!apt-get -qq install -y xtb

!xtb --version

# Download CREST.
# If the "latest" URL changes, replace this with a pinned release URL.
!wget -q -O /content/crest.tar.xz \
  "https://github.com/crest-lab/crest/releases/download/latest/crest-gnu-12-ubuntu-latest.tar.xz"

!rm -rf /content/crest
!tar -xf /content/crest.tar.xz -C /content
!chmod +x /content/crest/crest

!/content/crest/crest --version

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
      -----------------------------------------------------------      
     |                   =====================                   |     
     |                           x T B                           |     
     |                   =====================                   |     
     |                         S. Grimme                         |     
     |          Mulliken Center for Theoretical Chemistry        |     
     |                    University of Bonn                     |     
      -----------------------------------------------------------      

   * xtb version 6.6.1 (unknown) compiled by 'builduser@buildhost' on 2023-08-07

normal termination of xtb

       ╔════════════════════════════════════════════════╗
       ║                                                ║
       ║ 

## 2. Mount Google Drive

This will prompt you to authorize access, then your entire Drive is available at `/content/drive/MyDrive/...`

Put your 34 XYZ files in a folder in your Drive first (e.g. `MyDrive/TS_structures/`), then point
`DRIVE_FOLDER` below at it. Results (CSVs) will also be written back into that same folder so they
persist after the Colab runtime disconnects — anything saved only to `/content/...` is lost when the
session ends, so always write final outputs under `/content/drive/MyDrive/...`.

In [17]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import os
import numpy as np
import pandas as pd
import shutil
import subprocess
import re
import json

DRIVE_FOLDER = Path(
    "/content/drive/MyDrive/Cycloetherification_xyz_files"
)

RESULTS_FOLDER = DRIVE_FOLDER / "CREST_results"
RESULTS_FOLDER.mkdir(parents=True, exist_ok=True)

if not DRIVE_FOLDER.exists():
    raise FileNotFoundError(f"Drive folder does not exist: {DRIVE_FOLDER}")

print("Files found in DRIVE_FOLDER:")
for path in sorted(DRIVE_FOLDER.iterdir()):
    print(" ", path.name)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Files found in DRIVE_FOLDER:
  CREST_results
  R3_Control_Analysis
  R3_Control_Correlation_Plots
  RR_CH2-Ph.xyz
  RR_CH2-dioxane.xyz
  RR_CH2CN.xyz
  RR_CH2CO2Me.xyz
  RR_CMe2CO2Me.xyz
  RR_Cy.xyz
  RR_Et.xyz
  RR_Me-Propane.xyz
  RR_Me.xyz
  RR_Ph.xyz
  RR_alkene.xyz
  RR_alkyne.xyz
  RR_allyl.xyz
  RR_dimethyl-allyl.xyz
  RR_iPr.xyz
  RR_nBu.xyz
  RR_tBu.xyz
  RS_CH2-Ph.xyz
  RS_CH2-dioxane.xyz
  RS_CH2CN.xyz
  RS_CH2CO2Me.xyz
  RS_CMe2CO2Me.xyz
  RS_Cy.xyz
  RS_Et.xyz
  RS_Me-Propane.xyz
  RS_Me.xyz
  RS_Ph.xyz
  RS_alkene.xyz
  RS_alkyne.xyz
  RS_allyl.xyz
  RS_dimethyl-allyl.xyz
  RS_iPr.xyz
  RS_nBu.xyz
  RS_tBu.xyz
  superimposed
  superimposed_by_stereo


## 3. Pick a single structure to start with

(Either from Drive, or upload one directly if you just want to test the pipeline first.)

In [15]:
xyz_path = DRIVE_FOLDER / "RR_tBu.xyz"

if not xyz_path.exists():
    raise FileNotFoundError(f"XYZ file not found: {xyz_path}")

print("Using structure:", xyz_path)


Using structure: /content/drive/MyDrive/Cycloetherification_xyz_files/RR_tBu.xyz


## 4. Configuration

Set the **1-indexed** atom numbers (matching the order in your XYZ file) for:
- `ring_atoms`: the six ring atoms in connectivity order, e.g. `[O, C, C, C, C, C]`
- `forming_bond`: the two atoms whose distance should stay fixed during the search
  (usually the ring oxygen and the forming-bond carbon — same two atoms as the last
  entry in `ring_atoms`, i.e. `ring_atoms[0]` and `ring_atoms[-1]`)

Adjust these per structure — they do NOT have to be atoms 1-6; use whatever your
actual numbering is (e.g. 17-22 for the larger RR/RS scope structures).

In [16]:
ring_atoms = [17, 18, 19, 20, 21, 22]    # O-C-C-C-C-C in ring connectivity order
forming_bond = [17, 22]                  # atoms whose distance is held fixed

fc = 1.0                                 # constraint force constant (Hartree/Bohr^2)
method = "gfn2"                          # "gfn2" or "gff" (GFN-FF, faster/rougher)
quick_mode = True                        # True = faster/reduced sampling; False = thorough (slower)
threads = 2

# Six-torsion same-pucker threshold.
torsion_tolerance_deg = 5.0

CREST_EXECUTABLE = "/content/crest/crest"

if len(ring_atoms) != 6:
    raise ValueError("ring_atoms must contain exactly six atoms")

if len(forming_bond) != 2:
    raise ValueError("forming_bond must contain exactly two atoms")

if method not in {"gfn2", "gff"}:
    raise ValueError("method must be 'gfn2' or 'gff'")

## 5. Cremer–Pople ring-puckering tools

In [18]:
def parse_energy_from_comment(comment):
    """
    Parse an energy from an XYZ comment line.

    CREST comment formats may vary, so explicit energy patterns are
    checked before a general floating-point fallback.
    """
    patterns = [
        r"energy\s*=\s*([-+]?\d+(?:\.\d*)?(?:[Ee][-+]?\d+)?)",
        r"\bE\s*=\s*([-+]?\d+(?:\.\d*)?(?:[Ee][-+]?\d+)?)",
        r"([-+]?\d+\.\d+(?:[Ee][-+]?\d+)?)",
    ]

    for pattern in patterns:
        match = re.search(pattern, comment, flags=re.IGNORECASE)
        if match:
            return float(match.group(1))

    return np.nan

In [19]:
def read_xyz_frames(path):
    """
    Read a single- or multi-frame XYZ file.

    Returns a list of dictionaries containing atoms, coordinates,
    comment, and parsed energy.
    """
    path = Path(path)

    with path.open() as f:
        lines = f.readlines()

    frames = []
    position = 0

    while position < len(lines):
        if not lines[position].strip():
            position += 1
            continue

        try:
            n_atoms = int(lines[position].strip())
        except ValueError as exc:
            raise ValueError(
                f"Invalid atom-count line at {position + 1} in {path}"
            ) from exc

        end = position + n_atoms + 2

        if end > len(lines):
            raise ValueError(f"Incomplete XYZ frame in {path}")

        comment = lines[position + 1].rstrip("\n")
        atom_lines = lines[position + 2:end]

        atoms = []
        coords = []

        for line_number, line in enumerate(
            atom_lines,
            start=position + 3,
        ):
            parts = line.split()

            if len(parts) < 4:
                raise ValueError(
                    f"Invalid XYZ line {line_number} in {path}: {line}"
                )

            atoms.append(parts[0])
            coords.append(
                [
                    float(parts[1]),
                    float(parts[2]),
                    float(parts[3]),
                ]
            )

        frames.append(
            {
                "atoms": atoms,
                "coords": np.asarray(coords, dtype=float),
                "comment": comment,
                "energy_Eh": parse_energy_from_comment(comment),
            }
        )

        position = end

    return frames

In [20]:
def write_xyz_frames(frames, path):
    """Write one or more XYZ frames."""
    path = Path(path)

    with path.open("w") as f:
        for frame in frames:
            atoms = frame["atoms"]
            coords = frame["coords"]
            comment = frame.get("comment", "")

            f.write(f"{len(atoms)}\n")
            f.write(f"{comment}\n")

            for element, xyz in zip(atoms, coords):
                f.write(
                    f"{element:2s} "
                    f"{xyz[0]: .10f} "
                    f"{xyz[1]: .10f} "
                    f"{xyz[2]: .10f}\n"
                )

In [21]:
def validate_atom_indices(frame, indices, name):
    """Validate 1-indexed atom numbers."""
    n_atoms = len(frame["atoms"])

    if not indices:
        raise ValueError(f"{name} cannot be empty")

    if len(set(indices)) != len(indices):
        raise ValueError(f"{name} contains duplicate atom indices")

    if any(i < 1 or i > n_atoms for i in indices):
        raise IndexError(
            f"{name}={indices} is invalid for a structure with "
            f"{n_atoms} atoms"
        )

In [22]:
def distance_from_frame(frame, i, j):
    """Return a distance in Å using 1-indexed atom numbers."""
    validate_atom_indices(frame, [i, j], "distance_atoms")

    coords = frame["coords"]
    return float(np.linalg.norm(coords[i - 1] - coords[j - 1]))

## 6. Cremer–Pople and torsion functions

In [23]:
def cremer_pople_6(ring_coords):
    """
    Calculate Q, theta, and phi for a six-membered ring.

    ring_coords must contain six atoms in connectivity order.
    """
    R = np.asarray(ring_coords, dtype=float)

    if R.shape != (6, 3):
        raise ValueError(
            f"Expected ring coordinates with shape (6, 3), got {R.shape}"
        )

    centered = R - R.mean(axis=0)

    # Best-fit ring plane from SVD.
    _, _, vh = np.linalg.svd(centered, full_matrices=False)
    normal = vh[-1]

    z = centered @ normal
    j = np.arange(6)

    c = np.sqrt(2.0 / 6.0) * np.sum(
        z * np.cos(2.0 * np.pi * j / 3.0)
    )

    s = -np.sqrt(2.0 / 6.0) * np.sum(
        z * np.sin(2.0 * np.pi * j / 3.0)
    )

    q2 = np.sqrt(c**2 + s**2)
    q3 = np.sqrt(1.0 / 6.0) * np.sum(z * ((-1.0) ** j))

    Q = float(np.sqrt(q2**2 + q3**2))

    if Q < 1e-12:
        return Q, 0.0, 0.0

    theta = float(
        np.degrees(
            np.arccos(np.clip(q3 / Q, -1.0, 1.0))
        )
    )

    phi = float(np.degrees(np.arctan2(s, c)) % 360.0)

    return Q, theta, phi

In [24]:
def classify_6ring(theta, phi):
    """
    Heuristic six-membered-ring pucker classification.

    Treat structures close to a boundary cautiously.
    """
    theta = float(theta)
    phi = float(phi) % 360.0

    if theta < 15.0 or theta > 165.0:
        return "Chair (C)"

    if 75.0 <= theta <= 105.0:
        m = phi % 60.0
        return (
            "Boat (B)"
            if m < 15.0 or m > 45.0
            else "Twist-boat/Skew (S)"
        )

    if 35.0 <= theta <= 65.0 or 115.0 <= theta <= 145.0:
        m = phi % 60.0
        return (
            "Envelope (E)"
            if m < 15.0 or m > 45.0
            else "Half-chair (H)"
        )

    return "Intermediate"

In [25]:
def dihedral_angle(coords, i, j, k, l):
    """Calculate a dihedral angle in degrees using 0-indexed atoms."""
    p0, p1, p2, p3 = coords[[i, j, k, l]]

    b0 = -(p1 - p0)
    b1 = p2 - p1
    b2 = p3 - p2

    b1_norm = np.linalg.norm(b1)

    if b1_norm < 1e-12:
        raise ValueError("Central bond has zero length")

    b1 = b1 / b1_norm

    v = b0 - np.dot(b0, b1) * b1
    w = b2 - np.dot(b2, b1) * b1

    if np.linalg.norm(v) < 1e-12:
        raise ValueError("First projected vector is too small")

    if np.linalg.norm(w) < 1e-12:
        raise ValueError("Second projected vector is too small")

    x = np.dot(v, w)
    y = np.dot(np.cross(b1, v), w)

    return float(np.degrees(np.arctan2(y, x)))

In [26]:
def ring_torsions(frame, ring_atoms):
    """
    Calculate six sequential ring torsions.

    For [a0, a1, ..., a5], calculates:
        a0-a1-a2-a3
        a1-a2-a3-a4
        ...
        a5-a0-a1-a2
    """
    if len(ring_atoms) != 6:
        raise ValueError("ring_atoms must contain six atoms")

    validate_atom_indices(frame, ring_atoms, "ring_atoms")

    indices = [i - 1 for i in ring_atoms]
    coords = frame["coords"]

    torsions = []

    for shift in range(6):
        quartet = [
            indices[(shift + offset) % 6]
            for offset in range(4)
        ]

        torsions.append(dihedral_angle(coords, *quartet))

    return np.asarray(torsions, dtype=float)

In [27]:
def torsion_differences(torsions_a, torsions_b):
    """
    Return smallest absolute periodic differences in degrees.
    """
    a = np.asarray(torsions_a, dtype=float)
    b = np.asarray(torsions_b, dtype=float)

    if a.shape != b.shape:
        raise ValueError("Torsion vectors have different shapes")

    return np.abs((a - b + 180.0) % 360.0 - 180.0)

## 7. Run the constrained CREST conformer search (single structure)

In [28]:
def write_constraint_file(workdir, forming_bond, fc):
    """
    Write a harmonic bond-distance constraint in xTB format.

    The reference distance is taken from the input structure by xTB/CREST
    when no explicit reference coordinate is supplied.
    """
    constraint_text = f"""$constrain
distance: {forming_bond[0]}, {forming_bond[1]}, auto
force constant={fc}
$end
"""

    path = Path(workdir) / "constraints.inp"
    path.write_text(constraint_text)

    return path

In [29]:
def run_crest(
    xyz_file,
    workdir,
    forming_bond,
    fc=1.0,
    method="gfn2",
    quick_mode=True,
    threads=2,
):
    """Run one constrained CREST calculation."""
    xyz_file = Path(xyz_file)
    workdir = Path(workdir)
    workdir.mkdir(parents=True, exist_ok=True)

    input_frames = read_xyz_frames(xyz_file)

    if not input_frames:
        raise ValueError(f"No structure found in {xyz_file}")

    validate_atom_indices(
        input_frames[0],
        forming_bond,
        "forming_bond",
    )

    shutil.copy2(xyz_file, workdir / "struc.xyz")

    constraints_path = write_constraint_file(
        workdir=workdir,
        forming_bond=forming_bond,
        fc=fc,
    )

    cmd = [
        CREST_EXECUTABLE,
        "struc.xyz",
        "--cinp",
        constraints_path.name,
        "--T",
        str(int(threads)),
    ]

    if method == "gff":
        cmd.append("--gfnff")

    if quick_mode:
        cmd.append("-mquick")

    result = subprocess.run(
        cmd,
        cwd=workdir,
        capture_output=True,
        text=True,
        timeout=1800,
    )

    (workdir / "crest_stdout.log").write_text(result.stdout)
    (workdir / "crest_stderr.log").write_text(result.stderr)

    if result.returncode != 0:
        raise RuntimeError(
            f"CREST failed in {workdir}\n"
            f"Return code: {result.returncode}\n"
            f"Last stderr:\n{result.stderr[-3000:]}"
        )

    conformer_file = workdir / "crest_conformers.xyz"

    if not conformer_file.exists():
        raise FileNotFoundError(
            f"CREST finished but did not create {conformer_file}"
        )

    return conformer_file

## 8. Deduplicate by six ring torsions

In [30]:
def analyze_and_deduplicate(
    conformer_file,
    ring_atoms,
    forming_bond,
    torsion_tolerance_deg=5.0,
):
    """
    Analyze all CREST structures and retain one lowest-energy structure
    for each six-torsion ring-puckering class.
    """
    frames = read_xyz_frames(conformer_file)

    if not frames:
        raise ValueError(f"No conformers found in {conformer_file}")

    indexed_frames = list(enumerate(frames))

    indexed_frames.sort(
        key=lambda item: (
            not np.isfinite(item[1]["energy_Eh"]),
            item[1]["energy_Eh"]
            if np.isfinite(item[1]["energy_Eh"])
            else np.inf,
        )
    )

    retained = []
    retained_records = []
    duplicate_records = []

    for original_index, frame in indexed_frames:
        validate_atom_indices(frame, ring_atoms, "ring_atoms")
        validate_atom_indices(frame, forming_bond, "forming_bond")

        ring_coords = frame["coords"][
            [atom - 1 for atom in ring_atoms]
        ]

        Q, theta, phi = cremer_pople_6(ring_coords)
        torsions = ring_torsions(frame, ring_atoms)

        duplicate_of = None
        duplicate_diff = None

        for retained_item in retained:
            differences = torsion_differences(
                torsions,
                retained_item["torsions"],
            )

            if np.all(differences <= torsion_tolerance_deg):
                duplicate_of = retained_item["retained_conformer"]
                duplicate_diff = differences
                break

        record = {
            "original_conformer": original_index + 1,
            "energy_Eh": frame["energy_Eh"],
            "forming_bond_A": distance_from_frame(
                frame,
                forming_bond[0],
                forming_bond[1],
            ),
            "Q_A": Q,
            "theta_deg": theta,
            "phi_deg": phi,
            "classification": classify_6ring(theta, phi),
        }

        for index, torsion in enumerate(torsions, start=1):
            record[f"torsion_{index}_deg"] = torsion

        if duplicate_of is None:
            retained_conformer = len(retained) + 1

            retained.append(
                {
                    "frame": frame,
                    "torsions": torsions,
                    "retained_conformer": retained_conformer,
                }
            )

            record.update(
                {
                    "retained_conformer": retained_conformer,
                    "status": "retained",
                    "duplicate_of": np.nan,
                    "max_torsion_difference_deg": 0.0,
                    "mean_torsion_difference_deg": 0.0,
                }
            )

            retained_records.append(record)

        else:
            record.update(
                {
                    "retained_conformer": np.nan,
                    "status": "removed_torsion_duplicate",
                    "duplicate_of": duplicate_of,
                    "max_torsion_difference_deg": float(
                        np.max(duplicate_diff)
                    ),
                    "mean_torsion_difference_deg": float(
                        np.mean(duplicate_diff)
                    ),
                }
            )

            duplicate_records.append(record)

    retained_frames = [
        item["frame"]
        for item in retained
    ]

    retained_df = pd.DataFrame(retained_records)
    duplicates_df = pd.DataFrame(duplicate_records)

    if len(retained_df):
        minimum_energy = retained_df["energy_Eh"].min()

        if np.isfinite(minimum_energy):
            retained_df["rel_kcal"] = (
                retained_df["energy_Eh"] - minimum_energy
            ) * 627.5095
        else:
            retained_df["rel_kcal"] = np.nan

        retained_df = retained_df.sort_values(
            "rel_kcal",
            na_position="last",
        ).reset_index(drop=True)

    return retained_frames, retained_df, duplicates_df

## 9. Process the selected structure

In [31]:
def process_one_structure(
    xyz_file,
    results_folder,
    ring_atoms,
    forming_bond,
    fc=1.0,
    method="gfn2",
    quick_mode=True,
    threads=2,
    torsion_tolerance_deg=5.0,
):
    xyz_file = Path(xyz_file)
    results_folder = Path(results_folder)

    tag = xyz_file.stem
    workdir = Path("/content") / f"crest_run_{tag}"
    output_dir = results_folder / f"{tag}_CREST_conformations"

    output_dir.mkdir(parents=True, exist_ok=True)

    conformer_file = run_crest(
        xyz_file=xyz_file,
        workdir=workdir,
        forming_bond=forming_bond,
        fc=fc,
        method=method,
        quick_mode=quick_mode,
        threads=threads,
    )

    retained_frames, retained_df, duplicates_df = (
        analyze_and_deduplicate(
            conformer_file=conformer_file,
            ring_atoms=ring_atoms,
            forming_bond=forming_bond,
            torsion_tolerance_deg=torsion_tolerance_deg,
        )
    )

    # Preserve the raw CREST ensemble.
    shutil.copy2(
        conformer_file,
        output_dir / f"{tag}_raw_crest_conformers.xyz",
    )

    # Write each retained conformer separately.
    for frame, (_, row) in zip(
        retained_frames,
        retained_df.iterrows(),
    ):
        retained_number = int(row["retained_conformer"])
        energy = row["energy_Eh"]
        relative_energy = row["rel_kcal"]

        frame["comment"] = (
            f"retained_conformer={retained_number} "
            f"energy_Eh={energy:.12f} "
            f"rel_kcal={relative_energy:.6f}"
        )

        path = (
            output_dir
            / f"{tag}_retained_{retained_number:03d}.xyz"
        )

        write_xyz_frames([frame], path)

    # Write the filtered ensemble.
    if retained_frames:
        write_xyz_frames(
            retained_frames,
            output_dir / f"{tag}_retained_ensemble.xyz",
        )

    retained_csv = (
        output_dir
        / f"{tag}_retained_pucker_results.csv"
    )

    duplicate_csv = (
        output_dir
        / f"{tag}_torsion_duplicates.csv"
    )

    retained_df.to_csv(retained_csv, index=False)
    duplicates_df.to_csv(duplicate_csv, index=False)

    metadata = {
        "source_file": str(xyz_file),
        "ring_atoms": ring_atoms,
        "forming_bond": forming_bond,
        "constraint_force_constant": fc,
        "method": method,
        "quick_mode": quick_mode,
        "threads": threads,
        "torsion_tolerance_deg": torsion_tolerance_deg,
        "raw_conformer_count": len(
            read_xyz_frames(conformer_file)
        ),
        "retained_conformer_count": len(retained_frames),
        "removed_torsion_duplicate_count": len(duplicates_df),
    }

    with open(output_dir / f"{tag}_metadata.json", "w") as f:
        json.dump(metadata, f, indent=2)

    print(
        f"{tag}: retained {len(retained_frames)} of "
        f"{metadata['raw_conformer_count']} CREST structures; "
        f"removed {len(duplicates_df)} same-pucker duplicates"
    )

    return retained_df, duplicates_df

In [36]:
!pkill -f /content/crest/crest || true
!pkill -f xtb || true

^C
^C


In [37]:
!ps aux | grep -E "crest|xtb" | grep -v grep

In [38]:
from pathlib import Path

print("XYZ exists:", xyz_path.exists())
print("XYZ path:", xyz_path)

frames = read_xyz_frames(xyz_path)
print("Number of frames:", len(frames))
print("Number of atoms:", len(frames[0]["atoms"]))

validate_atom_indices(frames[0], ring_atoms, "ring_atoms")
validate_atom_indices(frames[0], forming_bond, "forming_bond")

print("Ring atoms:", ring_atoms)
print("Forming bond:", forming_bond)
print(
    "Initial forming-bond distance:",
    distance_from_frame(
        frames[0],
        forming_bond[0],
        forming_bond[1],
    ),
)

XYZ exists: True
XYZ path: /content/drive/MyDrive/Cycloetherification_xyz_files/RR_tBu.xyz
Number of frames: 1
Number of atoms: 56
Ring atoms: [17, 18, 19, 20, 21, 22]
Forming bond: [17, 22]
Initial forming-bond distance: 2.312109803551293


In [39]:
constraint_file = Path("/content/crest_run_RR_tBu/constraints.inp")

if constraint_file.exists():
    print(constraint_file.read_text())
else:
    print("Constraint file not found")

$constrain
distance: 17, 22, auto
force constant=1.0
$end



In [40]:
def diagnostic_crest(
    xyz_file,
    ring_atoms,
    forming_bond,
    method="gff",
    quick_mode=True,
    threads=2,
    timeout_seconds=300,
):
    import time

    xyz_file = Path(xyz_file)
    diagnostic_dir = Path("/content/crest_diagnostic")
    diagnostic_dir.mkdir(parents=True, exist_ok=True)

    shutil.copy2(xyz_file, diagnostic_dir / "struc.xyz")

    write_constraint_file(
        workdir=diagnostic_dir,
        forming_bond=forming_bond,
        fc=fc,
    )

    cmd = [
        CREST_EXECUTABLE,
        "struc.xyz",
        "--cinp",
        "constraints.inp",
        "--T",
        str(threads),
    ]

    if method == "gff":
        cmd.append("--gfnff")

    if quick_mode:
        cmd.append("-mquick")

    print("Command:")
    print(" ".join(cmd))
    print("\nCREST output:\n")

    start = time.time()

    process = subprocess.Popen(
        cmd,
        cwd=diagnostic_dir,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    try:
        for line in process.stdout:
            print(line, end="", flush=True)

            if time.time() - start > timeout_seconds:
                process.kill()
                process.wait()
                raise TimeoutError(
                    f"CREST exceeded {timeout_seconds} seconds"
                )

        return_code = process.wait()

    finally:
        if process.poll() is None:
            process.kill()
            process.wait()

    print("\nCREST return code:", return_code)

    if return_code != 0:
        raise RuntimeError(
            "CREST failed. Inspect the printed output above."
        )

    conformers = diagnostic_dir / "crest_conformers.xyz"

    if conformers.exists():
        print("Conformer file created:", conformers)
        print("Size:", conformers.stat().st_size, "bytes")
    else:
        print("No crest_conformers.xyz was created")

In [41]:
diagnostic_crest(
    xyz_file=xyz_path,
    ring_atoms=ring_atoms,
    forming_bond=forming_bond,
    method="gff",
    quick_mode=True,
    threads=2,
    timeout_seconds=300,
)

Command:
/content/crest/crest struc.xyz --cinp constraints.inp --T 2 --gfnff -mquick

CREST output:


       ╔════════════════════════════════════════════════╗
       ║                                                ║
       ║     ██████╗██████╗ ███████╗███████╗████████╗   ║
       ║    ██╔════╝██╔══██╗██╔════╝██╔════╝╚══██╔══╝   ║
       ║    ██║     ██████╔╝█████╗  ███████╗   ██║      ║
       ║    ██║     ██╔══██╗██╔══╝  ╚════██║   ██║      ║
       ║    ╚██████╗██║  ██║███████╗███████║   ██║      ║
       ║     ╚═════╝╚═╝  ╚═╝╚══════╝╚══════╝   ╚═╝      ║
       ║                                                ║
       ║    Conformer-Rotamer Ensemble Sampling Tool    ║
       ║            based on the xTB methods            ║
       ║                                                ║
       ╚════════════════════════════════════════════════╝
        Version 3.0.2, Tue, 16 June 10:18:10, 06/16/2026
        commit (cfdc301) compiled by 'usr@runnervm1li68'

   Cite work conducted with t

In [42]:
diagnostic_conformer_file = Path(
    "/content/crest_diagnostic/crest_conformers.xyz"
)

diagnostic_frames = read_xyz_frames(diagnostic_conformer_file)

print("Number of CREST frames:", len(diagnostic_frames))

forming_distances = [
    distance_from_frame(
        frame,
        forming_bond[0],
        forming_bond[1],
    )
    for frame in diagnostic_frames
]

print("Forming-bond distances:")
print("Minimum:", min(forming_distances), "Å")
print("Maximum:", max(forming_distances), "Å")
print("Mean:", np.mean(forming_distances), "Å")
print("Standard deviation:", np.std(forming_distances), "Å")

Number of CREST frames: 7
Forming-bond distances:
Minimum: 2.322638699855509 Å
Maximum: 2.324574542040689 Å
Mean: 2.323486098848155 Å
Standard deviation: 0.0007860863471054566 Å


In [44]:
from pathlib import Path
import numpy as np

diagnostic_conformer_file = Path(
    "/content/crest_diagnostic/crest_conformers.xyz"
)

if not diagnostic_conformer_file.exists():
    raise FileNotFoundError(
        f"Missing file: {diagnostic_conformer_file}"
    )

diagnostic_frames = read_xyz_frames(diagnostic_conformer_file)

if not diagnostic_frames:
    raise ValueError("No conformers were read from the CREST output")

input_frames = read_xyz_frames(xyz_path)

initial_distance = distance_from_frame(
    input_frames[0],
    forming_bond[0],
    forming_bond[1],
)

forming_distances = np.asarray(
    [
        distance_from_frame(
            frame,
            forming_bond[0],
            forming_bond[1],
        )
        for frame in diagnostic_frames
    ],
    dtype=float,
)

print("Number of CREST frames:", len(diagnostic_frames))
print(f"Initial forming-bond distance: {initial_distance:.6f} Å")
print()
print("Forming-bond distances in CREST ensemble:")
print(f"Minimum: {forming_distances.min():.6f} Å")
print(f"Maximum: {forming_distances.max():.6f} Å")
print(f"Mean:    {forming_distances.mean():.6f} Å")
print(f"Std:     {forming_distances.std(ddof=0):.6f} Å")
print(f"Range:   {np.ptp(forming_distances):.6f} Å")
print()
print("Maximum deviation from initial distance:")
print(
    f"{np.max(np.abs(forming_distances - initial_distance)):.6f} Å"
)

print()
print("Distance by conformer:")
for conformer_number, distance_value in enumerate(
    forming_distances,
    start=1,
):
    print(
        f"Conformer {conformer_number:3d}: "
        f"{distance_value:.6f} Å "
        f"(deviation "
        f"{distance_value - initial_distance:+.6f} Å)"
    )

Number of CREST frames: 7
Initial forming-bond distance: 2.312110 Å

Forming-bond distances in CREST ensemble:
Minimum: 2.322639 Å
Maximum: 2.324575 Å
Mean:    2.323486 Å
Std:     0.000786 Å
Range:   0.001936 Å

Maximum deviation from initial distance:
0.012465 Å

Distance by conformer:
Conformer   1: 2.323023 Å (deviation +0.010913 Å)
Conformer   2: 2.322862 Å (deviation +0.010753 Å)
Conformer   3: 2.322761 Å (deviation +0.010651 Å)
Conformer   4: 2.322639 Å (deviation +0.010529 Å)
Conformer   5: 2.324442 Å (deviation +0.012332 Å)
Conformer   6: 2.324575 Å (deviation +0.012465 Å)
Conformer   7: 2.324101 Å (deviation +0.011992 Å)


In [45]:
diagnostic_retained_frames, diagnostic_df, diagnostic_duplicates_df = (
    analyze_and_deduplicate(
        conformer_file=diagnostic_conformer_file,
        ring_atoms=ring_atoms,
        forming_bond=forming_bond,
        torsion_tolerance_deg=5.0,
    )
)

display(diagnostic_df)
display(diagnostic_duplicates_df)

,original_conformer,energy_Eh,forming_bond_A,Q_A,theta_deg,phi_deg,classification,torsion_1_deg,torsion_2_deg,torsion_3_deg,torsion_4_deg,torsion_5_deg,torsion_6_deg,retained_conformer,status,duplicate_of,max_torsion_difference_deg,mean_torsion_difference_deg,rel_kcal
0,1,-10.275028,2.323023,0.690704,97.152891,289.071567,Boat (B),64.752056,-59.872065,-5.694243,38.637675,-23.826879,-23.761996,1,retained,NaN,0.0,0.0,0.000000
1,3,-10.274495,2.322761,0.732248,92.557882,294.788491,Boat (B),59.954213,-61.845164,-6.071675,42.880970,-33.172509,-15.447986,2,retained,NaN,0.0,0.0,0.334990
2,5,-10.273248,2.324442,0.773649,86.959344,304.354479,Boat (B),49.979919,-64.356008,-1.641700,44.277850,-45.486229,-1.180593,3,retained,NaN,0.0,0.0,1.117563


,original_conformer,energy_Eh,forming_bond_A,Q_A,theta_deg,phi_deg,classification,torsion_1_deg,torsion_2_deg,torsion_3_deg,torsion_4_deg,torsion_5_deg,torsion_6_deg,retained_conformer,status,duplicate_of,max_torsion_difference_deg,mean_torsion_difference_deg
0,2,-10.274852,2.322862,0.707568,94.964572,291.911306,Boat (B),62.390247,-60.887301,-5.700919,40.469816,-28.176933,-19.781401,NaN,removed_torsion_duplicate,1,4.350054,2.257752
1,4,-10.274334,2.322639,0.750036,90.929661,296.653161,Boat (B),58.130527,-62.327355,-6.412061,44.518989,-36.648040,-12.355365,NaN,removed_torsion_duplicate,2,3.475532,1.808739
2,6,-10.273241,2.324575,0.781038,86.035223,305.919723,Boat (B),48.037383,-64.508216,-0.871406,44.349117,-47.503188,1.357579,NaN,removed_torsion_duplicate,3,2.538172,1.248573
3,7,-10.273180,2.324101,0.762200,88.421683,302.038113,Boat (B),52.793882,-64.078849,-2.573067,43.940616,-42.339174,-4.953950,NaN,removed_torsion_duplicate,3,3.773358,1.880023


In [46]:
print(diagnostic_df.columns.tolist())

['original_conformer', 'energy_Eh', 'forming_bond_A', 'Q_A', 'theta_deg', 'phi_deg', 'classification', 'torsion_1_deg', 'torsion_2_deg', 'torsion_3_deg', 'torsion_4_deg', 'torsion_5_deg', 'torsion_6_deg', 'retained_conformer', 'status', 'duplicate_of', 'max_torsion_difference_deg', 'mean_torsion_difference_deg', 'rel_kcal']


In [47]:
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 240)

display(
    diagnostic_df[
        [
            "original_conformer",
            "retained_conformer",
            "status",
            "duplicate_of",
            "energy_Eh",
            "rel_kcal",
            "forming_bond_A",
            "Q_A",
            "theta_deg",
            "phi_deg",
            "classification",
            "max_torsion_difference_deg",
            "mean_torsion_difference_deg",
        ]
    ]
)

,original_conformer,retained_conformer,status,duplicate_of,energy_Eh,rel_kcal,forming_bond_A,Q_A,theta_deg,phi_deg,classification,max_torsion_difference_deg,mean_torsion_difference_deg
0,1,1,retained,NaN,-10.275028,0.000000,2.323023,0.690704,97.152891,289.071567,Boat (B),0.0,0.0
1,3,2,retained,NaN,-10.274495,0.334990,2.322761,0.732248,92.557882,294.788491,Boat (B),0.0,0.0
2,5,3,retained,NaN,-10.273248,1.117563,2.324442,0.773649,86.959344,304.354479,Boat (B),0.0,0.0


In [49]:
display_df = diagnostic_df.copy()

display_df["energy_Eh"] = display_df["energy_Eh"].round(6)
display_df["rel_kcal"] = display_df["rel_kcal"].round(3)
display_df["forming_bond_A"] = display_df["forming_bond_A"].round(4)
display_df["Q_A"] = display_df["Q_A"].round(4)
display_df["theta_deg"] = display_df["theta_deg"].round(2)
display_df["phi_deg"] = display_df["phi_deg"].round(2)

display(
    display_df[
        [
            "original_conformer",
            "retained_conformer",
            "status",
            "energy_Eh",
            "rel_kcal",
            "forming_bond_A",
            "Q_A",
            "theta_deg",
            "phi_deg",
            "classification",
            "duplicate_of",
            "max_torsion_difference_deg",
            "mean_torsion_difference_deg",
        ]
    ]
)

,original_conformer,retained_conformer,status,energy_Eh,rel_kcal,forming_bond_A,Q_A,theta_deg,phi_deg,classification,duplicate_of,max_torsion_difference_deg,mean_torsion_difference_deg
0,1,1,retained,-10.275028,0.000,2.3230,0.6907,97.15,289.07,Boat (B),NaN,0.0,0.0
1,3,2,retained,-10.274495,0.335,2.3228,0.7322,92.56,294.79,Boat (B),NaN,0.0,0.0
2,5,3,retained,-10.273248,1.118,2.3244,0.7736,86.96,304.35,Boat (B),NaN,0.0,0.0


In [ ]:
single_df, single_duplicates_df = process_one_structure(
    xyz_file=xyz_path,
    results_folder=RESULTS_FOLDER,
    ring_atoms=ring_atoms,
    forming_bond=forming_bond,
    fc=fc,
    method=method,
    quick_mode=quick_mode,
    threads=threads,
    torsion_tolerance_deg=torsion_tolerance_deg,
)

single_df

## 10. Batch over all XYZ files

### 10.1 Batch settings

In [50]:
batch_method = "gff"
batch_quick_mode = True
batch_threads = 2
batch_timeout_seconds = 1800

torsion_tolerance_deg = 5.0

### 10.2 Atom-number overrides

In [51]:
OVERRIDES = {}

### 10.3 Validate all input files first

In [52]:
xyz_files = sorted(DRIVE_FOLDER.glob("*.xyz"))

print(f"Found {len(xyz_files)} XYZ files")

if len(xyz_files) == 0:
    raise FileNotFoundError(
        f"No XYZ files found in {DRIVE_FOLDER}"
    )

validation_records = []

for xyz_file in xyz_files:
    config = {
        "ring_atoms": list(ring_atoms),
        "forming_bond": list(forming_bond),
    }

    config.update(OVERRIDES.get(xyz_file.name, {}))

    try:
        frames = read_xyz_frames(xyz_file)

        if not frames:
            raise ValueError("No XYZ frames found")

        frame = frames[0]

        validate_atom_indices(
            frame,
            config["ring_atoms"],
            "ring_atoms",
        )

        validate_atom_indices(
            frame,
            config["forming_bond"],
            "forming_bond",
        )

        initial_distance = distance_from_frame(
            frame,
            config["forming_bond"][0],
            config["forming_bond"][1],
        )

        validation_records.append(
            {
                "structure": xyz_file.name,
                "n_atoms": len(frame["atoms"]),
                "input_frames": len(frames),
                "initial_forming_bond_A": initial_distance,
                "status": "OK",
                "error": "",
            }
        )

    except Exception as exc:
        validation_records.append(
            {
                "structure": xyz_file.name,
                "n_atoms": np.nan,
                "input_frames": np.nan,
                "initial_forming_bond_A": np.nan,
                "status": "FAILED",
                "error": repr(exc),
            }
        )

validation_df = pd.DataFrame(validation_records)

display(validation_df)

if (validation_df["status"] != "OK").any():
    raise RuntimeError(
        "Input validation failed. Fix the listed files before "
        "starting the batch."
    )

Found 34 XYZ files


,structure,n_atoms,input_frames,initial_forming_bond_A,status,error
0,RR_CH2-Ph.xyz,57,1,2.230903,OK,
1,RR_CH2-dioxane.xyz,56,1,2.309687,OK,
2,RR_CH2CN.xyz,48,1,2.237502,OK,
3,RR_CH2CO2Me.xyz,53,1,2.318575,OK,
4,RR_CMe2CO2Me.xyz,59,1,2.262033,OK,
5,RR_Cy.xyz,60,1,2.315953,OK,
6,RR_Et.xyz,50,1,2.254818,OK,
7,RR_Me-Propane.xyz,56,1,2.225446,OK,
8,RR_Me.xyz,47,1,2.303139,OK,
9,RR_Ph.xyz,54,1,2.284585,OK,


### 10.4 Batch execution with checkpointing

In [53]:
from pathlib import Path
import shutil
import json
import numpy as np
import pandas as pd

In [54]:
def structure_output_dir(results_folder, tag):
    return Path(results_folder) / f"{tag}_CREST_conformations"

In [55]:
def structure_is_complete(results_folder, tag):
    output_dir = structure_output_dir(results_folder, tag)

    required_files = [
        output_dir / f"{tag}_raw_crest_conformers.xyz",
        output_dir / f"{tag}_retained_ensemble.xyz",
        output_dir / f"{tag}_retained_pucker_results.csv",
        output_dir / f"{tag}_torsion_duplicates.csv",
        output_dir / f"{tag}_metadata.json",
    ]

    return all(path.exists() for path in required_files)

In [56]:
def process_one_structure_batch(
    xyz_file,
    results_folder,
    ring_atoms,
    forming_bond,
    fc=1.0,
    method="gff",
    quick_mode=True,
    threads=2,
    torsion_tolerance_deg=5.0,
    timeout_seconds=1800,
):
    """
    Run and process one XYZ structure.

    This function assumes that run_crest_live() and
    analyze_and_deduplicate() are already defined.
    """
    xyz_file = Path(xyz_file)
    results_folder = Path(results_folder)

    tag = xyz_file.stem
    workdir = Path("/content") / f"crest_batch_{tag}"
    output_dir = structure_output_dir(results_folder, tag)

    if workdir.exists():
        shutil.rmtree(workdir)

    workdir.mkdir(parents=True, exist_ok=True)
    output_dir.mkdir(parents=True, exist_ok=True)

    conformer_file = run_crest_live(
        xyz_file=xyz_file,
        workdir=workdir,
        forming_bond=forming_bond,
        fc=fc,
        method=method,
        quick_mode=quick_mode,
        threads=threads,
        timeout_seconds=timeout_seconds,
    )

    raw_frames = read_xyz_frames(conformer_file)

    if not raw_frames:
        raise ValueError(
            f"No conformers were found for {xyz_file.name}"
        )

    retained_frames, retained_df, duplicates_df = (
        analyze_and_deduplicate(
            conformer_file=conformer_file,
            ring_atoms=ring_atoms,
            forming_bond=forming_bond,
            torsion_tolerance_deg=torsion_tolerance_deg,
        )
    )

    # Preserve the raw CREST ensemble.
    shutil.copy2(
        conformer_file,
        output_dir / f"{tag}_raw_crest_conformers.xyz",
    )

    # Sort retained records by representative number before writing XYZ.
    if len(retained_df):
        retained_df = retained_df.sort_values(
            "retained_conformer"
        ).reset_index(drop=True)

    # Write retained ensemble.
    if retained_frames:
        write_xyz_frames(
            retained_frames,
            output_dir / f"{tag}_retained_ensemble.xyz",
        )

    # Write each retained structure individually.
    for frame, (_, row) in zip(
        retained_frames,
        retained_df.iterrows(),
    ):
        retained_number = int(row["retained_conformer"])

        energy = row["energy_Eh"]
        relative_energy = row["rel_kcal"]

        frame_copy = {
            "atoms": list(frame["atoms"]),
            "coords": np.array(frame["coords"], copy=True),
            "comment": (
                f"retained_conformer={retained_number} "
                f"energy_Eh={energy:.12f} "
                f"rel_kcal={relative_energy:.6f} "
                f"classification={row['classification']}"
            ),
        }

        write_xyz_frames(
            [frame_copy],
            output_dir
            / f"{tag}_retained_{retained_number:03d}.xyz",
        )

    # Save analysis tables.
    retained_df.to_csv(
        output_dir / f"{tag}_retained_pucker_results.csv",
        index=False,
    )

    duplicates_df.to_csv(
        output_dir / f"{tag}_torsion_duplicates.csv",
        index=False,
    )

    # Save metadata.
    metadata = {
        "structure": tag,
        "source_file": str(xyz_file),
        "ring_atoms": list(map(int, ring_atoms)),
        "forming_bond": list(map(int, forming_bond)),
        "constraint_force_constant": float(fc),
        "method": method,
        "quick_mode": bool(quick_mode),
        "threads": int(threads),
        "torsion_tolerance_deg": float(
            torsion_tolerance_deg
        ),
        "raw_conformer_count": int(len(raw_frames)),
        "retained_conformer_count": int(len(retained_frames)),
        "duplicate_count": int(len(duplicates_df)),
    }

    with open(output_dir / f"{tag}_metadata.json", "w") as f:
        json.dump(metadata, f, indent=2)

    return retained_df, duplicates_df

### 10.5 Run the batch

In [57]:
all_retained = []
all_duplicates = []
completed_records = []
failure_records = []

progress_path = RESULTS_FOLDER / "BATCH_progress.csv"
failure_path = RESULTS_FOLDER / "BATCH_failures.csv"
retained_path = RESULTS_FOLDER / "ALL_retained_pucker_results.csv"
duplicates_path = RESULTS_FOLDER / "ALL_torsion_duplicates.csv"

for file_number, xyz_file in enumerate(xyz_files, start=1):
    tag = xyz_file.stem

    print()
    print("=" * 80)
    print(
        f"[{file_number}/{len(xyz_files)}] "
        f"Processing: {xyz_file.name}"
    )
    print("=" * 80)

    config = {
        "ring_atoms": list(ring_atoms),
        "forming_bond": list(forming_bond),
    }

    config.update(OVERRIDES.get(xyz_file.name, {}))

    try:
        if structure_is_complete(RESULTS_FOLDER, tag):
            print(f"Already complete; loading saved results: {tag}")

            output_dir = structure_output_dir(
                RESULTS_FOLDER,
                tag,
            )

            retained_df = pd.read_csv(
                output_dir / f"{tag}_retained_pucker_results.csv"
            )

            duplicates_df = pd.read_csv(
                output_dir / f"{tag}_torsion_duplicates.csv"
            )

        else:
            retained_df, duplicates_df = (
                process_one_structure_batch(
                    xyz_file=xyz_file,
                    results_folder=RESULTS_FOLDER,
                    ring_atoms=config["ring_atoms"],
                    forming_bond=config["forming_bond"],
                    fc=fc,
                    method=batch_method,
                    quick_mode=batch_quick_mode,
                    threads=batch_threads,
                    torsion_tolerance_deg=torsion_tolerance_deg,
                    timeout_seconds=batch_timeout_seconds,
                )
            )

        if len(retained_df):
            retained_copy = retained_df.copy()
            retained_copy.insert(0, "structure", tag)
            all_retained.append(retained_copy)

        if len(duplicates_df):
            duplicates_copy = duplicates_df.copy()
            duplicates_copy.insert(0, "structure", tag)
            all_duplicates.append(duplicates_copy)

        completed_records.append(
            {
                "structure": tag,
                "status": "completed",
                "raw_or_retained_rows": len(retained_df),
                "duplicate_rows": len(duplicates_df),
                "error": "",
            }
        )

        # Save checkpoint tables after every successful structure.
        if all_retained:
            pd.concat(
                all_retained,
                ignore_index=True,
            ).to_csv(retained_path, index=False)

        if all_duplicates:
            pd.concat(
                all_duplicates,
                ignore_index=True,
            ).to_csv(duplicates_path, index=False)

        pd.DataFrame(completed_records).to_csv(
            progress_path,
            index=False,
        )

        print(
            f"Completed: {tag} | "
            f"retained={len(retained_df)} | "
            f"duplicates={len(duplicates_df)}"
        )

    except Exception as exc:
        failure_record = {
            "structure": tag,
            "status": "failed",
            "error": repr(exc),
        }

        failure_records.append(failure_record)
        completed_records.append(failure_record)

        pd.DataFrame(completed_records).to_csv(
            progress_path,
            index=False,
        )

        pd.DataFrame(failure_records).to_csv(
            failure_path,
            index=False,
        )

        print(f"FAILED: {tag}")
        print(repr(exc))


[1/34] Processing: RR_CH2-Ph.xyz
FAILED: RR_CH2-Ph
NameError("name 'run_crest_live' is not defined")

[2/34] Processing: RR_CH2-dioxane.xyz
FAILED: RR_CH2-dioxane
NameError("name 'run_crest_live' is not defined")

[3/34] Processing: RR_CH2CN.xyz
FAILED: RR_CH2CN
NameError("name 'run_crest_live' is not defined")

[4/34] Processing: RR_CH2CO2Me.xyz
FAILED: RR_CH2CO2Me
NameError("name 'run_crest_live' is not defined")

[5/34] Processing: RR_CMe2CO2Me.xyz
FAILED: RR_CMe2CO2Me
NameError("name 'run_crest_live' is not defined")

[6/34] Processing: RR_Cy.xyz
FAILED: RR_Cy
NameError("name 'run_crest_live' is not defined")

[7/34] Processing: RR_Et.xyz
FAILED: RR_Et
NameError("name 'run_crest_live' is not defined")

[8/34] Processing: RR_Me-Propane.xyz
FAILED: RR_Me-Propane
NameError("name 'run_crest_live' is not defined")

[9/34] Processing: RR_Me.xyz
FAILED: RR_Me
NameError("name 'run_crest_live' is not defined")

[10/34] Processing: RR_Ph.xyz
FAILED: RR_Ph
NameError("name 'run_crest_live' is

### 10.6 Save final combined tables

In [58]:
if all_retained:
    all_retained_df = pd.concat(
        all_retained,
        ignore_index=True,
    )
else:
    all_retained_df = pd.DataFrame()

if all_duplicates:
    all_duplicates_df = pd.concat(
        all_duplicates,
        ignore_index=True,
    )
else:
    all_duplicates_df = pd.DataFrame()

failures_df = pd.DataFrame(failure_records)

all_retained_df.to_csv(
    RESULTS_FOLDER / "ALL_retained_pucker_results.csv",
    index=False,
)

all_duplicates_df.to_csv(
    RESULTS_FOLDER / "ALL_torsion_duplicates.csv",
    index=False,
)

failures_df.to_csv(
    RESULTS_FOLDER / "ALL_batch_failures.csv",
    index=False,
)

print()
print("Batch finished.")
print(f"Retained rows: {len(all_retained_df)}")
print(f"Duplicate rows: {len(all_duplicates_df)}")
print(f"Failed structures: {len(failures_df)}")
print("Results folder:", RESULTS_FOLDER)


Batch finished.
Retained rows: 0
Duplicate rows: 0
Failed structures: 34
Results folder: /content/drive/MyDrive/Cycloetherification_xyz_files/CREST_results


In [59]:
if len(all_retained_df):
    summary_df = (
        all_retained_df
        .groupby(
            ["structure", "classification"],
            dropna=False,
        )
        .size()
        .reset_index(name="n_retained")
        .sort_values(["structure", "classification"])
    )

    display(summary_df)

In [60]:
%notebook -e all_code.py
